In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skewnorm
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
!pip install openai
import openai

openai.api_key = key

In [ ]:
# https://www.kaggle.com/datasets/joebeachcapital/top-2000-companies-globally?resource=download
df_leads = pd.read_csv(dir + '/Top2000CompaniesGlobally.csv', usecols=['Company', 'Country'])

# Shuffle
df_leads = df_leads.sample(frac=1, random_state=93).reset_index(drop=True)

# Display
df_leads.head(10)

# Read Leads Source

In [ ]:
df_leads = pd.read_csv(dir + '/Leads.csv')
df_leads.columns = df_leads.columns.str.lower()   # Make column names lowercase
num_rows = len(df_leads)

# Shuffle
df_leads = df_leads.sample(frac=1, random_state=29).reset_index(drop=True)

# Generate IDs
ids = [f'ACC{i+1:04d}' for i in range(num_rows)]
# Insert the IDs as a new column at the beginning of the DataFrame
df_leads.insert(0, 'id', ids)

print(f'Total Leads: {num_rows}')
display(df_leads.head(10))

In [ ]:
df_leads = df_leads[['id', 'company_name', 'industry', 'mock_source'
                    ]]

# Lead Dates
Assumptions
1. Present day is 5/1/2025.
2. Assume a max lead lifecycle of 12 months.
3. Assume a right skew distribution of lead lifecycle. (Lead drop-off. More leads in beginning, less towards end of lifecycle)
4. Assume a historical conversion rate of around 30%.
5. Assume an even rate of conversion throughout a year
6. Assume a lead snapshot is created every 2 weeks.

Lead lifecycle may be bimodal, with many leads closing early due to non-interest in beginning, and some may last until the end of the lifecycle, stagnating until sunset. For simplicity, we will assume right skew.

A closed, converted lead should have had a lifecycle of at least 2-3 months to account for time. All others may terminate at any stage, say after 1 month.

Steps
1. Assign all actual customer accounts to converted (Closed)
2. Randomly assign other accounts to Closed or Open at 80/20. Of closed leads, randomly assign to Converted at 30% rate.
3. Randomly simulate closing dates for closed leads, ranging from 12 months prior to present. Simulate the lead lifecycle to determine opening date within 12 months to 3 months prior to closing.
4. Randomly simulate the lead lifecycle to determine opening date within 12 months before present.  
5. For each lead, create a lead snapshot every 2 weeks.

Basically, an open lead is any lead that is open as of present day. Its opening date could be anywhere from today to 12 months ago, erring towards more recent. Closed leads have closed anytime within the last 12 months.


In [ ]:
# Set assumptions
present_date = pd.to_datetime('2025-05-01')
max_lead_months = 12
close_p = 0.8
convert_p = 0.3

## Assign Status

In [ ]:
# Assign all actual customer leads to Closed
df_leads_customer = df_leads[df_leads['mock_source'] == 'customer']
df_leads_customer.loc[:, 'status'] = 'Closed'

In [ ]:
# Assign other leads to Closed at 80% rate
df_leads_other = df_leads[df_leads['mock_source'] != 'customer']

# Generate a random selection for each row in the DataFrame
rng = np.random.RandomState(55)
random_status = rng.choice(['Closed', 'Open'], size=len(df_leads_other), p=[close_p, 1-close_p])

df_leads_other.loc[:, 'status'] = random_status

In [ ]:
# Concatenate
df_leads = pd.concat([df_leads_customer, df_leads_other])
# Reset index
df_leads = df_leads.reset_index(drop=True)

In [ ]:
# Apply conditions for 'Converted' column based on 'Status' and 'mock_source'
df_leads.loc[df_leads['status'] == 'Closed', 'converted'] = 0  # Initialize 'Converted' to 0 for Closed status

# For Closed leads where mock_source is customer, set Converted to 1
df_leads.loc[(df_leads['status'] == 'Closed') & (df_leads['mock_source'] == 'customer'), 'converted'] = 1

# For Closed leads where mock_source is not customer, apply Converted=1 at 30% rate
closed_non_customer_leads = df_leads[(df_leads['status'] == 'Closed') & (df_leads['mock_source'] != 'customer')]
random_conversion = rng.choice([1, 0], size=len(closed_non_customer_leads), p=[convert_p, 1-convert_p])
df_leads.loc[(df_leads['status'] == 'Closed') & (df_leads['mock_source'] != 'customer'), 'converted'] = random_conversion

df_leads

## Simulate Dates

In [ ]:
# Set a close date for each closed lead
df_closed_leads = df_leads[df_leads['status'] == 'Closed']
rng = np.random.RandomState(43)
time_deltas = rng.randint(0, 365, size=len(df_closed_leads))  # closed within 365 days
close_dates = present_date - pd.to_timedelta(time_deltas, unit='D')
df_leads.loc[df_leads['status'] == 'Closed', 'close_date'] = close_dates

1. Assume a closed, converted lead should take at least 3 months to progress through all stages.
2. Assume a closed, lost lead, could take as little as 1 month.
3. Open leads can be however new.

In [ ]:
# Simulate lead lifecycle distribution
# Assume similar distribution for all regardless of converted status.
# Converted leads minimum 3 months. All else between 1 and 12.

close_convert_dist = skewnorm.rvs(a=4, loc=120, scale=(60), size=1000)
close_convert_dist = np.clip(close_convert_dist, 90, 365)

close_lost_dist = skewnorm.rvs(a=4, loc=120, scale=(60), size=1000)
close_lost_dist = np.clip(close_lost_dist, 30, 365)

open_dist = skewnorm.rvs(a=4, loc=120, scale=(60), size=1000)
open_dist = np.clip(open_dist, 1, 365)

# Create a figure with 3 subplots side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # 1 row, 3 columns

# Plot Close Convert Distribution
axes[0].hist(close_convert_dist, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Close Convert Distribution')
axes[0].set_xlabel('Lifecycle Days')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', alpha=0.5)

# Plot Close Lost Distribution
axes[1].hist(close_lost_dist, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Close Lost Distribution')
axes[1].set_xlabel('Lifecycle Days')
axes[1].set_ylabel('Frequency')
axes[1].grid(axis='y', alpha=0.5)

# Plot Open Distribution
axes[2].hist(open_dist, bins=30, edgecolor='black', alpha=0.7)
axes[2].set_title('Open Distribution')
axes[2].set_xlabel('Open Days')
axes[2].set_ylabel('Frequency')
axes[2].grid(axis='y', alpha=0.5)

# Adjust layout to prevent overlapping titles/labels
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
def simulate_lifecycle(row):
    # Simulate lifecycle in days
    if row['status'] == 'Closed' and row['converted'] == 1:
      days = skewnorm.rvs(a=4, loc=120, scale=(60), size=1)[0]
      days = int(np.clip(days, 90, 365))
      open_date = row['close_date'] - pd.Timedelta(days=days)
    elif row['status'] == 'Closed' and row['converted'] == 0:
      days = skewnorm.rvs(a=4, loc=120, scale=(60), size=1)[0]
      days = int(np.clip(days, 30, 365))
      open_date = row['close_date'] - pd.Timedelta(days=days)
    else: # status == 'Open'
      days = skewnorm.rvs(a=4, loc=120, scale=(60), size=1)[0]
      days = int(np.clip(days, 1, 365))
      open_date = present_date - pd.Timedelta(days=days)

    return open_date

In [ ]:
np.random.seed(86)
df_leads['open_date'] = df_leads.apply(simulate_lifecycle, axis=1)
# Reorder last 2 colunms
cols = df_leads.columns[:-2].tolist() + ['open_date', 'close_date']
df_leads = df_leads[cols]

df_leads

## Visualize Timeline

In [ ]:
# Set temporary close date for visualization
df_leads_temp = df_leads.copy()
df_leads_temp = df_leads_temp.sample(n=150, random_state=54) # Sample for readability
df_leads_temp['close_date'] = df_leads_temp['close_date'].fillna(present_date)    # Temporary close date for now
df_leads_temp = df_leads_temp.sort_values(by='open_date', ascending=False).reset_index(drop=True)   # Sort by Open Date


colors = {'Open': 'blue', 'Closed': 'black'}

# Create the figure and axes
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the open and close dates as lines
for index, row in df_leads_temp.iterrows():
    open_date = row['open_date']
    close_date = row['close_date']
    status = row['status']
    color = colors.get(status, 'orange')  # Default to blue if status not in colors

    # Plot a line segment from Open Date to Close Date
    ax.plot([open_date, close_date], [index, index], marker='o', markersize=3, linestyle='-', color=color, label=status)

# Add a vertical line at 2025-05-01
vertical_date = pd.to_datetime('2025-05-01')
ax.axvline(vertical_date, color='red', linestyle='--', label='May 1, 2025 (Present)')

# Set x-axis label and format
ax.set_xlabel('Date')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45, ha='right')

# Remove y-axis ticks and labels
ax.set_yticks([])
ax.set_yticklabels([])
ax.set_ylabel(None)  # Optional: Remove the y-axis label entirely

# Set title
plt.title('Lead Lifecycles')

# Add legend (only show one label per status)
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), title='Status')

# Adjust layout to prevent labels from overlapping
plt.tight_layout()

# Show the plot
plt.show()

# Lead Profile

In [ ]:
np.random.seed(23) # Setting a new random seed

## Summary of Company
The information here is potentially based on:
- Lead enrichment
- Sales organization research and engagement

The idea is that this is still a relatively automated process that is not part of the primary lead prioritization workflow.

In [ ]:
def generate_summary(row):

    company_name = row['company_name']
    industry = row['industry']

    prompt = f"""
    Write a natural-sounding company summary.

    Company: {company_name}
    Industry: {industry}

    Include
    - What the company does and specializes in
    - Its estimated scale by number of employees or annual revenue, or both (Use specific numbers or ranges)
    - Its market position (e.g., startup, growing, established, leader)
    - Speculate on the company’s technology stack — including its data and digital maturity, presence of high value data, and its use of cloud platforms

    Avoid using bullet points, emojis, labels, or section headers.

    If exact information is unavailable, invent plausible-sounding details.
    """

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()


df_leads['company_summary'] = df_leads.apply(generate_summary, axis=1)

In [ ]:
df_leads

## Assign favorability

In [ ]:
# Assign Favorability Score
def get_favorability(row):

    if row['status'] == 'Closed' and row['converted'] == 1:
        favorability = np.random.randint(70, 101)  # Favorable score estimate
    elif row['status'] == 'Closed' and row['converted'] == 0:
        favorability = np.random.randint(1, 91)  # Unfavorable score estimate
    elif row['status'] == 'Open':
        favorability = np.random.randint(1, 101)  # Broad favorability estimate

    return favorability


df_leads['favorability'] = df_leads.apply(get_favorability, axis=1)

## Investment and Competitor Solution
Shopping behavior. Company Cyber Defense investment low, medium, or high. High investment means they are well-aware and budgeting for cyber defense. Also consider the degree of cyber resilience current business. If in business, when is renewal date?

In [ ]:
def assign_investment_level(row):
    favorability = row['favorability']
    # investment mapping
    mapping = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}

    if favorability < 20:
        p = [0.5, 0.3, 0.2, 0.0]
    elif favorability < 50:
        p = [0.3, 0.4, 0.2, 0.1]
    elif favorability < 70:
        p = [0.1, 0.4, 0.4, 0.1]
    elif favorability < 85:
        p = [0.0, 0.3, 0.4, 0.3]
    else:  # favorability >= 85
        p = [0.0, 0.1, 0.4, 0.5]

    # Select investment level from appropriate distribution
    investment_num = np.random.choice([1, 2, 3, 4], p=p)

    return mapping[investment_num]


df_leads['cyber_investment'] = df_leads.apply(assign_investment_level, axis=1)

- Among medium and higher investments: 40% with competitor solution, 40% without, 20% unknown
- Among low investments: 10% with competitor solution, 70% without, 20% unknown.
- With competitor solution, between present and 24 months in future for renewal.
- Mask some information later on

In [ ]:
# Active with competitor solution?
def assign_competitor_solution(row):
    if row['cyber_investment'] != 'low':
        return np.random.choice(
            ['Competitor Solution', 'No Solution', 'Unknown'],
            p=[0.4, 0.4, 0.2]
        )
    else:
        return np.random.choice(
            ['Competitor Solution', 'No Solution', 'Unknown'],
            p=[0.1, 0.7, 0.2]
        )

# If so renewal date?
def assign_renewal_date(row):
    # More Favorable 6-12 months from open
    if row['competitor_solution'] == 'Competitor Solution' and row['favorability'] > 75:
        start_date = row['open_date'] + pd.DateOffset(months=6)
        end_date = row['open_date'] + pd.DateOffset(months=12)
        return  pd.to_datetime(np.random.uniform(present_date.value, end_date.value)).normalize()
    # Favorable 3-15 months from open
    elif row['competitor_solution'] == 'Competitor Solution' and row['favorability'] > 50:
        start_date = row['open_date'] + pd.DateOffset(months=3)
        end_date = row['open_date'] + pd.DateOffset(months=15)
        return  pd.to_datetime(np.random.uniform(present_date.value, end_date.value)).normalize()
    elif row['competitor_solution'] == 'Competitor Solution' and row['favorability'] <= 50:
    # Unfavorable 1-30 months from open
        start_date = row['open_date']
        end_date = present_date + pd.DateOffset(months=30)
        return  pd.to_datetime(np.random.uniform(present_date.value, end_date.value)).normalize()

df_leads['competitor_solution'] = df_leads.apply(assign_competitor_solution, axis=1)
df_leads['renewal_date'] = df_leads.apply(assign_renewal_date, axis=1)

## Deal Value
Simulate deal value based on cyber investment and favorability

In [ ]:
investment_multiplier = {
    'Low': 0.6,
    'Medium': 0.8,
    'High': 1.0,
    'Very High': 1.2
}

# Generate noisy, non-linear deal value
def generate_deal_value(row):

    # Calculate investment factor based on cyber investment
    investment_factor = investment_multiplier.get(row['cyber_investment'])

    favorability = row['favorability'] / 100  # normalize to 0–1

    # Use a log-normal distribution to model deal values
    mu = np.log(150000)  # Shift the mean towards the higher end of the deal range (150K)
    sigma = 0.6  # control how wide the distribution is (lower = narrower, higher = wider)

    # Random deal value generation using log-normal distribution
    log_normal_value = np.random.lognormal(mu, sigma)

    # Amplify favorability effect: raising favorability to a higher power pushes high favorability more
    favorability_factor = favorability ** 2.5  # Exponent increases the impact of favorability

    # Scale by favorability and investment
    scaled_value = log_normal_value * investment_factor * favorability_factor

    # Clip the value only if it's extreme (do this last to allow most variation)
    if scaled_value < 50000:
        return 50000 + np.random.normal(0, 5000)  # Add a little noise if it's too low
    elif scaled_value > 300000:
        return 300000 - np.random.normal(0, 5000)  # Add a little noise if it's too high
    else:
        return scaled_value


df_leads['deal_value'] = df_leads.apply(generate_deal_value, axis=1)
# Round to nearest 1000
df_leads['deal_value'] = (df_leads['deal_value'] / 1000).round() * 1000
df_leads['deal_value'] = df_leads['deal_value'].astype(int)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df_leads is your DataFrame containing the deal_value column
# You can replace df_leads['deal_value'] with your actual column if necessary

# Plotting histogram
plt.figure(figsize=(5, 3))
sns.histplot(df_leads['deal_value'], bins=50, kde=True, color='skyblue', edgecolor='black')

# Adding labels and title
plt.title('Distribution of Deal Values')
plt.xlabel('Deal Value')
plt.ylabel('Frequency')

# Show the plot
plt.show()

df_leads.plot(kind='scatter', x='favorability', y='deal_value', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)
plt.show()

In [ ]:
df_leads

# Snapshots

Snapshot every 2 or 4 weeks from open date, and final snapshot at close date if applicable.

For all closed leads, first snapshot is Prospecting, last snapshot is Convert or Lost.

Between can be any of the stages from prospecting to negotiation, but must follow that order. Converted leads should be more likely to reach the later stages sooner, whereas lost leads should not, and may not make it far at all.


In [ ]:
np.random.seed(94940) # Setting a new random seed

In [ ]:
def create_snapshots(row):
    """Creates recurring snapshots for a given lead."""
    snapshots = []

    # Assign snapshot bounds
    start_date = row['open_date']
    if row['status'] == 'Closed':
        end_date = row['close_date']
    else: end_date = present_date   # Assign present date as proxy close date if lead is open

    snapshot_date = row['open_date']
    snapshot_seq = 1
    while snapshot_date < end_date:
        # Create a dict of the row to represent a snapshot
        snapshot = row.to_dict()
        # Add date and sequence number
        snapshot['snapshot_date'] = snapshot_date
        snapshot['snapshot_seq'] = snapshot_seq

        if snapshot_date == start_date:   # Snapshot on open date
            snapshot['lead_stage'] = 'Prospecting'
        elif start_date < snapshot_date < end_date:    # Snapshot between open and close date
            snapshot['lead_stage'] = None

        # Add snapshot to list
        snapshots.append(snapshot)
        # Increment
        snapshot_date += pd.DateOffset(days=14)
        snapshot_seq += 1

    # Add snapshot for close date (Not technically a lead anymore if closed date)
    snapshot = row.to_dict()
    snapshot['snapshot_date'] = end_date
    snapshot['snapshot_seq'] = snapshot_seq
    # Determined if lead is closed, None if open
    if row['status'] == 'Closed' and row['converted'] == 1:
      snapshot['lead_stage'] = 'Converted'
    elif row['status'] == 'Closed' and row['converted'] == 0:
      snapshot['lead_stage'] = 'Lost'
    else: # status is Open
      snapshot['lead_stage'] = None

    # Add snapshot to list
    snapshots.append(snapshot)

    return snapshots

## Simulate Pipeline Progression

In [ ]:
pipeline_stages = ['Prospecting', 'Qualified', 'Proposal', 'Negotiation', 'Converted/Lost']

def simulate_pipeline_progression(snapshots):
    """Takes in snapshots for one lead and assigns lead stages over time."""
    n_snapshots = len(snapshots)    # Count of snapshots
    # Get lead results
    lead_status = snapshots[0]['status']
    lead_converted = snapshots[0]['converted']
    favorability = snapshots[0]['favorability']

    if lead_status == 'Closed' and lead_converted == 1:
        # favorability = np.random.randint(70, 101)  # Favorable score estimate
        stages = pipeline_stages  # Progress through all stages

    elif lead_status == 'Closed' and lead_converted == 0:
        # favorability = np.random.randint(0, 91)  # Unfavorable score estimate
        max_stage_idx = np.searchsorted([20, 50, 70, 85], favorability, side='right')
        stages = pipeline_stages[:max_stage_idx + 1] # May not progress through all stages

    elif lead_status == 'Open':
        # favorability = np.random.randint(0, 101)  # Broad favorability estimate
        max_stage_idx = np.searchsorted([20, 50, 70, 85], favorability, side='right')
        stages = pipeline_stages[:max_stage_idx + 1] # Never reach Converted/Lost determination. Remove closed stage if present

    # Now assign stages evenly across snapshots
    stage_per_snapshot = []
    stages = [s for s in stages if s != 'Converted/Lost'] # Remove closed stage if present, add only on final
    n_stages = len(stages)
    for idx, snapshot in enumerate(snapshots):
        is_final = (idx == n_snapshots - 1)  # Check if last snapshot
        snapshot = snapshot.copy()  # Copy row

        if is_final and lead_status == 'Closed' and snapshot['converted'] == 1:
            snapshot['lead_stage'] = 'Converted' # Only on the last snapshot
        elif is_final and lead_status == 'Closed' and snapshot['converted'] == 0:
            snapshot['lead_stage'] = 'Lost' # Only on the last snapshot
        else:
            stage_idx = min(idx * n_stages // (n_snapshots - 1), n_stages - 1)
            snapshot['lead_stage'] = stages[stage_idx]

        stage_per_snapshot.append(snapshot)

    return stage_per_snapshot

## Simulate Engagement and Shopping Behavior

Correlate engagement with favorability and lead stage.
Decision-makers (DM): Staff, Director, Executive (% Chance of CIO or CISO)

Downloads, Website visits, Emails, Highest decision-maker involved

Correlate recent events with cybersecurity investment. DM involvement is also depending on investment.

In [ ]:
stage_logic = {
    'Prospecting': {
        'downloads': (0, 2),
        'website_visits': (0, 4),
        'interactions': (0, 4),
        'dm_probabilities': [0.90, 0.10, 0.00, 0.00],
    },
    'Qualified': {
        'downloads': (1, 2),
        'website_visits': (0, 4),
        'interactions': (4, 8),
        'dm_probabilities': [0.75, 0.20, 0.05, 0.00],
    },
    'Proposal': {
        'downloads': (1, 4),
        'website_visits': (2, 8),
        'interactions': (4, 10),
        'dm_probabilities': [0.10, 0.60, 0.20, 0.10],
    },
    'Negotiation': {
        'downloads': (1, 4),
        'website_visits': (2, 8),
        'interactions': (8, 12),
        'dm_probabilities': [0.00, 0.50, 0.30, 0.20],
    }
}
dm_levels = ["Manager", "Director", "VP", "Exec/CIO/CISO"]


def scale_random(min_val, max_val, favorability):
    """
    Generate a favorability-weighted random value with light noise.
    """
    # Map favorability to 0-1
    favorability = favorability / 100
    # Calculate value based on favorability, then add noise
    base = min_val + (max_val - min_val) * favorability
    noise = np.random.normal(0, (max_val - min_val) * 0.1)
    # Clip within bounds
    return int(np.clip(base + noise, min_val, max_val))


def choose_dm_level(prev_level_index, dm_probs):
    # Sample a DM level ≥ previous level.
    probs = np.array(dm_probs) / np.sum(dm_probs)  # Normalize probabilities
    # Randomly get an index based on probabilities
    new_level_index = np.random.choice(np.arange(len(dm_probs)), p=probs)
    # Advance only if the new index is greater than the previous
    final_level_index = max(prev_level_index, new_level_index)

    dm_level = dm_levels[final_level_index]
    return dm_level


def simulate_engagement(snapshots):
    # Simulates engagement
    previous_dm_index = defaultdict(lambda: 0)  # Per lead_id

    for snapshot in snapshots:
        lead_id = snapshot['id']
        stage = snapshot['lead_stage']
        favorability = snapshot['favorability']
        cyber_investment = snapshot['cyber_investment']

        # Handle Lost and Converted stages
        if stage in ['Converted', 'Lost']:
            snapshot['downloads'] = 0
            snapshot['website_visits'] = 0
            snapshot['interactions'] = 1
            snapshot['decision_maker_level'] = dm_levels[prev_dm]
            continue

        # Get rules
        logic = stage_logic[stage]

        # Generate values
        snapshot['downloads'] = scale_random(*logic['downloads'], favorability)   # Provide bounds and favorability
        snapshot['website_visits'] = scale_random(*logic['website_visits'], favorability)
        snapshot['interactions'] = scale_random(*logic['interactions'], favorability)

        # Decision-maker level
        prev_dm = previous_dm_index[lead_id]
        new_dm = choose_dm_level(prev_dm, logic['dm_probabilities'])
        previous_dm_index[lead_id] = max(prev_dm, dm_levels.index(new_dm))
        snapshot['decision_maker_level'] = new_dm

    return snapshots

## Simulate Discvoery
Discovery indexes based on assumption of equal stage pipeline times. Blanket assumption for leads that do not reach end of pipeline.

In [ ]:
def delay_discovery(snapshots):
    # Extract the sequence length
    seq_len = len(snapshots)

    # Investment/Competitor Discovery - before proposal
    min_idx = 0
    max_idx = int(seq_len * 0.5)
    # Normal distribution
    mu = (min_idx + max_idx) / 2
    sigma = max(1, (max_idx - min_idx) / 4)  # 95% within range
    idx1 = int(np.clip(np.random.normal(mu, sigma), min_idx, max_idx - 1))

    # Deal Value Discovery - before negotiation
    min_idx = int(seq_len * 0.25)
    max_idx = int(seq_len * 0.75)
    # Normal distribution
    mu = (min_idx + max_idx) / 2
    sigma = max(1, (max_idx - min_idx) / 4)  # 95% within range
    idx2 = int(np.clip(np.random.normal(mu, sigma), min_idx, max_idx - 1))

    # Masking
    for i in range(seq_len):
        if i < idx1:
            snapshots[i]['cyber_investment'] = 'Unknown'
            snapshots[i]['competitor_solution'] = 'Unknown'
        if i < idx2:
            snapshots[i]['deal_value'] = 'Unknown'

    return snapshots

The later the stage, the more confident in deal value. Becomes more clear towards the end of qualified stage, pretty clear in proposal and negotiation stage.

In [ ]:
all_snapshots = []
for idx, row in df_leads.iterrows():
    # Creat snapshots, one account per loop
    lead_snapshots = create_snapshots(row)

    # Simulate lead stages per snapshot per lead
    simulated_snapshots = simulate_pipeline_progression(lead_snapshots)

    # Simulate engagement. Snapshots must be input in order
    simulated_snapshots = simulate_engagement(simulated_snapshots)

    # Delay discovery
    simulated_snapshots = delay_discovery(simulated_snapshots)

    # Append engagement snapshots
    all_snapshots.extend(simulated_snapshots)


df_snapshots = pd.DataFrame(all_snapshots)
df_snapshots

## Visualize Pipeline

In [ ]:
# Sample 100 random unique leads
sampled_leads = df_snapshots['id'].drop_duplicates().sample(n=20, random_state=649)
# Filter the snapshots to just these leads
df_snapshots_temp = df_snapshots[df_snapshots['id'].isin(sampled_leads)]
# Sort by date
df_snapshots_temp = df_snapshots_temp.sort_values(by='open_date', ascending=False).reset_index(drop=True)

# Example stage -> color mapping
stage_colors = {
    'Prospecting': 'gray',
    'Qualified': 'dodgerblue',
    'Proposal': 'blue',
    'Negotiation': 'navy',
    'Converted': 'green',
    'Lost': 'red',
    }

# Create figure
fig, ax = plt.subplots(figsize=(14, 8))

# Ensure snapshot_date is datetime
df_snapshots_temp['snapshot_date'] = pd.to_datetime(df_snapshots_temp['snapshot_date'])

# Get unique leads
leads = df_snapshots_temp['id'].unique()

# Map each lead to a y-position
lead_y_positions = {lead: i for i, lead in enumerate(leads)}

# Plot each snapshot
for _, row in df_snapshots_temp.iterrows():
    lead_id = row['id']
    x = row['snapshot_date']
    y = lead_y_positions[lead_id]
    stage = row['lead_stage']
    color = stage_colors.get(stage, 'black')  # fallback to black if missing

    ax.scatter(x, y, color=color, label=stage, s=100, edgecolor='k')

# Draw a horizontal line for each lead
for lead_id, y in lead_y_positions.items():
    lead_snapshots = df_snapshots_temp[df_snapshots_temp['id'] == lead_id]
    min_date = lead_snapshots['snapshot_date'].min()
    max_date = lead_snapshots['snapshot_date'].max()
    ax.plot([min_date, max_date], [y, y], color='lightgray', linestyle='--')

# Formatting
ax.set_xlabel('Snapshot Date')
ax.set_title('Lead Pipeline Progression Over Time')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)

# Remove y-axis ticks and labels
ax.set_yticks([])
ax.set_yticklabels([])
ax.set_ylabel(None)  # Optional: Remove the y-axis label entirely

# Remove duplicate labels in legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title='Pipeline Stage')

plt.tight_layout()
plt.show()

# Export

In [ ]:
df_snapshots = df_snapshots.sort_values(by=['id', 'snapshot_date']).reset_index(drop=True)
df_snapshots

In [ ]:
df_snapshots.decision_maker_level.value_counts()

In [ ]:
# Export
df_snapshots.to_csv(dir + '/lead_snapshots.csv', index=False)